# Analysis

Compare model performance on food nutrition estimation.

In [25]:
import pandas as pd
import numpy as np
import re
from pathlib import Path
from sklearn.metrics import r2_score, mean_absolute_error

In [26]:
DATA_DIR = Path('data')
RESULTS_DIR = DATA_DIR / 'results'
COLS = ['calories', 'protein_g', 'fat_g', 'carbs_g']

## Load benchmark results

In [27]:
# auto-discover all result files
results = {}
for f in sorted(RESULTS_DIR.glob('*.json')):
    model_name = f.stem
    df = pd.read_json(f)
    for c in COLS:
        df[f'pred_{c}'] = pd.to_numeric(df[f'pred_{c}'], errors='coerce')
    # infer soft vs raw: raw prompts match "Xg of ..." pattern
    df['is_soft'] = ~df['prompt'].str.match(r'^\d+g of ')
    results[model_name] = df

print(f"Found {len(results)} model results: {list(results.keys())}")
# check split
sample = list(results.values())[0]
print(f"Soft: {sample['is_soft'].sum()}, Raw: {(~sample['is_soft']).sum()}")

Found 4 model results: ['claude-haiku-4.5', 'claude-sonnet-4.6', 'gemini-3-flash-preview', 'qwen3-235b-a22b-2507']
Soft: 2497, Raw: 2503


## Scoring & metrics

In [28]:
def score_model(df: pd.DataFrame, col: str) -> dict:
    """Compute R2, MAE, and MAPE for a single column."""
    mask = df[f'pred_{col}'].notna() & df[col].notna()
    y_true = df.loc[mask, col]
    y_pred = df.loc[mask, f'pred_{col}']
    
    if len(y_true) < 2:
        return {'r2': np.nan, 'mae': np.nan, 'mape': np.nan, 'n': len(y_true)}
    
    r2 = r2_score(y_true, y_pred)
    mae = mean_absolute_error(y_true, y_pred)
    # MAPE: avoid div by zero, skip rows where true == 0
    nonzero = y_true != 0
    mape = np.mean(np.abs((y_true[nonzero] - y_pred[nonzero]) / y_true[nonzero])) * 100 if nonzero.sum() > 0 else np.nan
    
    return {'r2': round(r2, 4), 'mae': round(mae, 1), 'mape': round(mape, 1), 'n': int(mask.sum())}

In [29]:
def build_table(results, split_name='all'):
    """Build one table: models as rows, R2 per column + mean as columns."""
    split_fn = SPLITS[split_name]
    
    rows = []
    for model_name, df in results.items():
        sdf = split_fn(df)
        row = {'model': model_name}
        r2s = []
        for col in COLS:
            s = score_model(sdf, col)
            row[col] = s['r2']
            r2s.append(s['r2'])
        row['mean_r2'] = round(np.nanmean(r2s), 4)
        rows.append(row)
    
    return pd.DataFrame(rows).set_index('model').sort_values('mean_r2', ascending=False)

In [30]:
for split in ['all', 'soft', 'raw']:
    print(f"\n=== {split.upper()} (R²) ===")
    display(build_table(results, split))


=== ALL (R²) ===


,calories,protein_g,fat_g,carbs_g,mean_r2
model,,,,,
gemini-3-flash-preview,0.8806,0.8930,0.8736,0.8164,0.8659
claude-sonnet-4.6,0.8273,0.7747,0.8253,0.7823,0.8024
claude-haiku-4.5,0.6371,0.6205,0.7459,0.6861,0.6724
qwen3-235b-a22b-2507,0.6664,0.5876,0.6200,0.6314,0.6264



=== SOFT (R²) ===


,calories,protein_g,fat_g,carbs_g,mean_r2
model,,,,,
gemini-3-flash-preview,0.6718,0.8383,0.4534,0.7469,0.6776
claude-sonnet-4.6,0.6991,0.6927,0.6376,0.6366,0.6665
claude-haiku-4.5,0.6277,0.5535,0.5477,0.6474,0.5941
qwen3-235b-a22b-2507,0.2208,0.4453,-0.2554,0.6452,0.2640



=== RAW (R²) ===


,calories,protein_g,fat_g,carbs_g,mean_r2
model,,,,,
gemini-3-flash-preview,0.9046,0.9281,0.9122,0.8103,0.8888
claude-sonnet-4.6,0.8293,0.8227,0.8340,0.7821,0.8170
qwen3-235b-a22b-2507,0.7051,0.6685,0.6905,0.6015,0.6664
claude-haiku-4.5,0.5914,0.6453,0.7493,0.6663,0.6631


Expect soft results to not be as good as information is ambiguous from data (deliberatly to mimic human data formats). Raw data is a more format benchmark and results are generally pretty accurate across models. 